# PCA on preprocessed data


    Imports

In [59]:
import os
import random

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import time

In [60]:
SEED = 0          # même graine que 1-Prepocessing : découpage identique
D = 50            # dimension de représentation, IDENTIQUE pour la PCA et l'AE
TEST_SIZE = 0.2   # part de cellules en validation
BATCH_SIZE = 128

In [64]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.width", 200)

repertoire = os.getcwd()
DATA_DIR = os.path.dirname(repertoire)

Data loading

In [ ]:
train = sc.read_h5ad(f"{DATA_DIR}/train.h5ad")
val = sc.read_h5ad(f"{DATA_DIR}/val.h5ad")

Xt = train.X
Xv = val.X

if sp.issparse(Xt):
    X_train = Xt.toarray()
else:
    X_train = Xt
X_train = np.asarray(X_train, dtype = np.float32)

if sp.issparse(Xv):
    X_val = Xv.toarray()
else:
    X_val = Xv
X_val = np.asarray(X_val, dtype = np.float32)

print(X_train)
print(X_train.shape)
print(X_val.shape)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/autofs/unitytravail/travail/tgonzalezbon/data/train.h5ad', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

PCA

In [ ]:
pca = PCA(n_components=D, random_state=SEED)

Z_pca_train = pca.fit_transform(X_train)   # (n_train, 50)
Z_pca_val = pca.transform(X_val)           # (n_val, 50)

X_pca_train = pca.inverse_transform(Z_pca_train)
X_pca_val = pca.inverse_transform(Z_pca_val)

mse_pca_train = mean_squared_error(X_train, X_pca_train)
mse_pca_val = mean_squared_error(X_val, X_pca_val)

print(f"variance expliquée par {D} composantes : {pca.explained_variance_ratio_.sum():.3f}")
print(f"MSE PCA - train : {mse_pca_train:.4f} | validation : {mse_pca_val:.4f}")

PCA Visualsation

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=adata.obs["celltype"].astype("category").cat.codes,
    cmap="viridis",
    s=5
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA - Cell types")
plt.colorbar(label="Cell type")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    range(1, 51),
    np.cumsum(pca.explained_variance_ratio_),
    marker="o"
)

plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA - Explained variance")
plt.grid()
plt.show()

Data reconstruction

In [ ]:
X_reconstructed = pca.inverse_transform(X_pca)

Reconstructed_pca = pca.transform(X_reconstructed)

Reconstructed data visualisation

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    Reconstructed_pca[:, 0],
    Reconstructed_pca[:, 1],
    c=adata.obs["celltype"].astype("category").cat.codes,
    cmap="viridis",
    s=5
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA - Cell types")
plt.colorbar(label="Cell type")
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(X[0], label="Original")
plt.plot(X_reconstructed[0], label="Reconstructed")

plt.legend()
plt.xlabel("Genes")
plt.ylabel("Expression")
plt.title("Original vs Reconstructed")

In [ ]:
plt.figure(figsize=(8, 5))

plt.imshow(X - X_reconstructed, aspect="auto", cmap="coolwarm")
plt.colorbar(label="Difference")

plt.xlabel("Genes")
plt.ylabel("Cells")
plt.title("Reconstruction error")

plt.show()

Reconstruction evaluation

In [ ]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(X, X_reconstructed)

print("Reconstruction error:", mse)

In [ ]:
import os
import numpy as np

# Créer un dossier pour les résultats s'il n'existe pas
os.makedirs("results", exist_ok=True)

# Sauvegarder les résultats de la PCA
np.savez_compressed(
    "results/resultats_PCA.npz",

    Z_train=Z_pca_train,
    Z_val=Z_pca_val,

    mse_train=mse_pca_train,
    mse_val=mse_pca_val,

    variance_expliquee=pca.explained_variance_ratio_.sum(),

    # Identification des cellules
    train_obs_names=train.obs_names.to_numpy(),
    val_obs_names=val.obs_names.to_numpy()
)

print("Résultats PCA sauvegardés.")